# Molecular diffusion of dissolved CO2

This notebook reproduces Section 6 of `report/report.tex` (the paper's
Fig. 6): a FluidFlower-inspired convective mixing problem, run at two grid
resolutions and two diffusivities.

## Model and numerical diffusion

When CO2 dissolves into brine, the mixture is denser than the resident
brine and sinks in Rayleigh-Taylor fingers, which accelerates dissolution
trapping. Concentration gradients also drive a diffusive mass flux. The
total macroscale flux of component $\gamma$ in phase $\alpha$ decomposes
as
$$
\mathbf{J}^\gamma_\alpha =
\underbrace{\rho_\alpha\,\chi^\gamma_\alpha\,\mathbf{u}_\alpha}_{\text{advective}}
\;-\;
\underbrace{\phi\,\rho_\alpha S_\alpha\, D^\gamma_\alpha\,
\nabla\chi^\gamma_\alpha}_{\text{diffusive (Fickian)}},
$$
where $\chi^\gamma_\alpha$ is the mass fraction and $D^\gamma_\alpha$ a
scalar pseudo-diffusivity; for CO2 in brine the molecular value is
$\mathcal{D}\sim2$-$5\times10^{-9}\,\mathrm{m^2/s}$. MRST implements this
by replacing the `ComponentTotalFlux` state function of
`GenericBlackOilModel` with `CO2TotalFluxWithDiffusion`.

Whether the diffusive term matters depends on the grid. A first-order
upstream scheme carries a numerical diffusion $D_{\mathrm{num}}\sim u\,h$,
with $u$ the Darcy velocity and $h$ the cell size. For this problem
($k=100$ mD, $\mu=0.8$ cP, pressure gradient 10 mbar/m),
$u\approx1.2\times10^{-7}$ m/s, so $h=1$ cm gives
$D_{\mathrm{num}}\sim10^{-9}\,\mathrm{m^2/s}$ — larger than the physical
diffusivity — while $h=2.5$ mm brings the two within reach of each other.
Physical diffusion therefore only shapes the solution once the grid is fine
enough; a coarse grid already contains more numerical diffusion than the
physics requires.

## A FluidFlower-inspired convective mixing problem

The test case is a quasi-2D stratigraphic section ($1\times0.66$ m, one
1-cm-thick cell layer) inspired by the FluidFlower benchmark rig,
repositioned at 1 km depth so that CO2 is supercritical. The PEBI mesh
conforms to a fault structure (fault permeability 10 mD inside a 1 D
reservoir); sealing units are removed from the flow domain, and lateral
boundary cells receive a $10^5$ pore-volume multiplier to emulate an open
aquifer. CO2 is injected in the lower-right reservoir at 8 mL/min (surface
conditions) for one day, with rate ramps, then the model runs to $t=30$
days with dissolution active.

We compare two meshes (coarse, $h\approx1$ cm, 8,206 cells; fine,
$h\approx2.5$ mm, 105,279 cells) at two diffusivities each
($D=0$ and $D=2\times10^{-11}\,\mathrm{m^2/s}$): four simulations in
total.

In [ ]:
run('/home/adriano/codes/MRST/startup.m');
mrstModule add upr coarsegrid co2lab-mit ad-props ad-blackoil deckformat ad-core
addpath('/home/adriano/codes/MRST/reproductions/salo2024-gcs/scripts/shim');   % delaunayTriangulation
addpath('/home/adriano/codes/MRST/reproductions/salo2024-gcs/scripts');        % computeWellIndexPalagiAziz
addpath('/home/adriano/codes/MRST/core/utils/octave_only/mrst');              % fastInterpTable
mrstVerbose off

RESDIR = '/home/adriano/codes/MRST/reproductions/salo2024-gcs/results';
FIGDIR = '/home/adriano/codes/MRST/reproductions/salo2024-gcs/figures';
FLUIDDECK = ['/home/adriano/codes/MRST/mrst-adblackoil-gcs/diffusion/' ...
             'input_files_diff/fluid_props/example_co2brine_1kmDepth_3regions.DATA'];

## A note on mesh generation and Octave

`simpleExtrudedFluidFlowerMesh` builds the 2-D PEBI grid with the `upr`
module, extrudes it, and **caches** the result under
`core/output/generated_meshes/`. On this machine, the `coarse` and `fine`
meshes at 1 km depth are already cached from an earlier session, so the
cells below hit the cache and do not touch `upr`'s grid generator at all.

This matters because `upr`'s `sortEdges.m` has a malformed block comment
that Octave (unlike MATLAB) parses as an unterminated comment, silently
returning grids unsorted and corrupting the extrusion (see
[the portability notes in the README](../README.md)). The fix lives on
branch `fix/upr-octave-block-comments` (upstream PR #37, not yet merged).
**You only need to check out that branch if you request a mesh resolution
that is not already cached** (`coarse` and `fine` are; e.g. `medium` is
not) or pass `force_regen=true`. The cells below never do either by
default.

## Mesh preview

The mesh itself, before any simulation: the 2-D PEBI grid (`G2D`, before
extrusion to the single 1-cm-thick cell layer), colored by the compartments
defined in `run_diffusion_case` above (gray: sealing units removed from the
flow domain; blue: reservoir; orange: the fault, permeability 10 mD; red:
the injection well cell). Loading both resolutions here only reads the
existing cache (a few seconds), regardless of the `RUN_LIVE` flags
below.

In [ ]:
sealID = [1 3 5 8];
faultID = 6;

G_coarse = simpleExtrudedFluidFlowerMesh('coarse');
G_fine   = simpleExtrudedFluidFlowerMesh('fine');

meshes = {G_coarse, G_fine};
titles = {sprintf('Coarse mesh (%d cells)', G_coarse.G2D.cells.num), ...
          sprintf('Fine mesh (%d cells)', G_fine.G2D.cells.num)};

figure('Position', [50 50 1200 500]);
for k = 1:2
    Gd = meshes{k};
    isSeal  = ismember(Gd.compartID, sealID);
    isFault = Gd.compartID == faultID;
    isRes   = ~isSeal & ~isFault;

    subplot(1,2,k);
    plotGrid(Gd.G2D, isSeal,  'facecolor', [0.75 0.75 0.75], 'edgecolor', 'none');
    hold on
    plotGrid(Gd.G2D, isRes,   'facecolor', [0.75 0.85 1.00], 'edgecolor', 'none');
    plotGrid(Gd.G2D, isFault, 'facecolor', [1.00 0.55 0.35], 'edgecolor', 'none');
    plotGrid(Gd.G2D, logical(Gd.G2D.cells.tag), 'facecolor', 'r', 'edgecolor', 'r');
    hold off
    axis equal tight, box on
    title(titles{k});
    xlabel('x [m]'); ylabel('z = depth [m]');
end
print(gcf, fullfile(FIGDIR, 'fig_diffusion_mesh.png'), '-dpng', '-r130');

### PEBI cells, zoomed near the fault

The panels above show the whole domain; drawing every cell edge at that
scale is unreadable (105,279 edges for the fine mesh) and slow. Zooming
into a 20x15 cm window around the fault-reservoir crossing near the
injection well shows the actual PEBI polygons, and why the fine mesh
resolves the convective fingers of Section 6.2 while the coarse mesh does
not.

In [ ]:
zoom_x = [0.43 0.63];
zoom_z = [0.28 0.43];

figure('Position', [50 50 1200 500]);
for k = 1:2
    Gd = meshes{k};
    G2Dg = computeGeometry(Gd.G2D);   % G2D has no centroids until this is called
    cc = G2Dg.cells.centroids;
    inZoom = cc(:,1)>=zoom_x(1) & cc(:,1)<=zoom_x(2) & ...
             cc(:,2)>=zoom_z(1) & cc(:,2)<=zoom_z(2);
    isSeal  = ismember(Gd.compartID, sealID) & inZoom;
    isFault = (Gd.compartID == faultID) & inZoom;
    isRes   = inZoom & ~isSeal & ~isFault;

    subplot(1,2,k);
    plotGrid(Gd.G2D, isSeal,  'facecolor', [0.75 0.75 0.75], 'edgecolor', 'k');
    hold on
    plotGrid(Gd.G2D, isRes,   'facecolor', [0.75 0.85 1.00], 'edgecolor', 'k');
    plotGrid(Gd.G2D, isFault, 'facecolor', [1.00 0.55 0.35], 'edgecolor', 'k');
    plotGrid(Gd.G2D, logical(Gd.G2D.cells.tag) & inZoom, 'facecolor', 'r', 'edgecolor', 'r');
    hold off
    axis equal tight, box on
    xlim(zoom_x); ylim(zoom_z);
    title(sprintf('%s cells in view', num2str(sum(inZoom))));
    xlabel('x [m]'); ylabel('z = depth [m]');
end
print(gcf, fullfile(FIGDIR, 'fig_diffusion_mesh_zoom.png'), '-dpng', '-r130');

In [ ]:
RUN_LIVE_COARSE = false;   # ~167 s + ~167 s = ~5.6 min for the pair
RUN_LIVE_FINE   = false;   # ~59 min + ~46 min = ~1.75 h for the pair

In [ ]:
function run_diffusion_case(meshres, Denv, outtag, resdir, fluiddeck, wellCell, wiModel)
    % wellCell (optional): cell index to inject into. Default: nearest to
    % the paper's [0.005, 0.903, 1000.58] location (see Appendix).
    % wiModel  (optional): 'peaceman' (default, MRST's own) or
    % 'palagi-aziz' (see Appendix) -- only changes how the well index of
    % wellCell is computed; everything else is identical.
    if nargin < 6, wellCell = []; end
    if nargin < 7 || isempty(wiModel), wiModel = 'peaceman'; end

    G_dat = simpleExtrudedFluidFlowerMesh(meshres);
    assert(all(G_dat.G.cells.volumes > 0), 'Mesh has non-positive cell volumes');
    assert(all(G_dat.G.faces.areas > 0), 'Mesh has non-positive face areas');
    fprintf('Mesh %s: %d cells, min volume %.3e\n', meshres, ...
            G_dat.G.cells.num, min(G_dat.G.cells.volumes));
    sealID = [1 3 5 8];
    faultID = 6;
    G = G_dat.G;
    if isfield(G.faces, 'tag'), G.faces = rmfield(G.faces, 'tag'); end
    [G, cellmap] = removeCells(G, ismember(G_dat.compartID, sealID));

    cid = 1:G_dat.G.cells.num;
    fid = cid(G_dat.compartID == faultID);
    fid = ismember(cellmap, fid);
    rock.poro = 0.3*ones(G.cells.num, 1);
    rock.poro(fid) = 0.1;
    rock.perm = 1000*ones(G.cells.num, 1);
    rock.perm(fid) = 10;
    rock.perm = rock.perm*(milli*darcy);
    rock.regions.saturation = ones(G.cells.num, 1);
    rock.regions.saturation(fid) = 3;
    rock.regions.rocknum = ones(G.cells.num, 1);

    deck = convertDeckUnits(readEclipseDeck(fluiddeck));
    deck.REGIONS.ROCKNUM = rock.regions.rocknum;
    fluid = initDeckADIFluid(deck);

    gravity reset on
    g = norm(gravity);
    water_column = 1000;
    p_r = 1*barsa + g*fluid.rhoOS*water_column;
    z_0 = min(G.cells.centroids(:,3));
    z_max = max(G.cells.centroids(:,3));
    nz = 2000;
    zv = linspace(z_0, z_max, nz)';
    dz = zv(2) - zv(1);
    ph = zeros(nz,1);  ph(1) = p_r;
    for k = 2:nz
        dpdz = g * fluid.bO(ph(k-1), 0, false) * fluid.rhoOS;
        pmid = ph(k-1) + 0.5*dz*dpdz;
        ph(k) = ph(k-1) + dz * g * fluid.bO(pmid, 0, false) * fluid.rhoOS;
    end
    p0 = interp1(zv, ph, G.cells.centroids(:,3));
    s0  = repmat([1, 0], [G.cells.num, 1]);
    rs0 = zeros(G.cells.num, 1);
    rv0 = 0;
    state0 = struct('s', s0, 'rs', rs0, 'rv', rv0, 'pressure', p0);

    t = [60*minute 1*day 30*day];
    reportTimes = [(12:12:t(1)/minute)*minute, ...
                   (2:1:24)*hour, ...
                   (1440+5:5:1465)*minute, ...
                   ([25 26 28 32 36 40 48 60 72 96 120])*hour, ...
                    (6:30)*day];
    wellno = 1;
    rate   = 8;                                      % mL/min (surface conditions)
    injrate = rate*(milli*litre)/(minute*wellno);     % Sm3/s
    if isempty(wellCell)
        dist = sqrt(sum((G.cells.centroids - [0.005, 0.903, 1000.58]).^2, 2));
        [~, wellInx] = min(dist);
    else
        wellInx = wellCell;
    end
    radius = 1e-3;
    if strcmpi(wiModel, 'palagi-aziz')
        WI = computeWellIndexPalagiAziz(G, rock, wellInx, radius, 'Dir', 'z');
    else
        WI = computeWellIndex(G, rock, radius, wellInx, 'Dir', 'z');
    end
    W = addWell([ ], G, rock, wellInx, 'Name', 'I1', 'Dir', 'z', ...
                'WI', WI, ...
                'Type', 'rate', 'Val', injrate, 'compi', [0, 1], ...
                'refDepth', G.cells.centroids(wellInx, G.griddim), ...
                'Radius', 1e-3);
    timesteps = [reportTimes(1) diff(reportTimes)];

    model = GenericBlackOilModel(G, rock, fluid, 'disgas', true, 'water', false);
    model.minimumPressure = min(state0.pressure);
    model = model.validateModel();

    if Denv > 0
        diffFlux = CO2TotalFluxWithDiffusion(model);
        diffFlux.componentDiffusion = [0 Denv];
        diffFlux.faceAverage = true;
        model.FlowDiscretization.ComponentTotalFlux = diffFlux;
    end

    nls = getNonLinearSolver(model, 'TimestepStrategy', 'iteration');
    nls.LinearSolver = BackslashSolverAD();
    nls.useLinesearch = true;
    nls.maxIterations = 10;
    nls.maxTimestepCuts = 12;
    nls.acceptanceFactor = 2;

    L = max(G.faces.centroids(:,2));
    f = any([G.faces.centroids(:,2) == 0, ...
             G.faces.centroids(:,2) > L-1e-3], 2);
    cellsext = unique(reshape(G.faces.neighbors(f, :), [], 1));
    cellsext(cellsext==0) = [];
    model.operators.pv(cellsext) = model.operators.pv(cellsext)*10^5;
    bc = [];

    schedule_inj = simpleSchedule(timesteps, 'W', W, 'bc', bc);
    n_ramp = 5;
    v = injrate;
    injrates = [0.01*v 0.1*v 0.2*v 0.5*v v ...
                0.8*v 0.5*v 0.1*v 0.01*v 0];
    tmp = cell(numel(injrates), 1);
    schedule = struct('step', schedule_inj.step);
    schedule.control = struct('W', tmp, 'bc', tmp, 'src', tmp);
    for n=1:numel(injrates)
        schedule.control(n).W = W;
        schedule.control(n).W.val = injrates(n);
        schedule.control(n).bc = bc;
    end
    idStep = find(cumsum(schedule.step.val) < t(1));
    schedule.step.control(idStep) = 1:max(idStep);
    schedule.step.control(idStep(end)+1:end) = max(idStep)+1;
    idStep2 = find(cumsum(schedule.step.val) > t(2), 1);
    schedule.step.control(idStep2:idStep2+(n_ramp -1)) = (1:n_ramp) + n_ramp;
    schedule.step.control(idStep2+n_ramp:end) = 2*n_ramp;

    t_start = tic;
    [wellSols, states, report] = simulateScheduleAD(state0, model, schedule, ...
                                                    'NonLinearSolver', nls);
    t_elapsed = toc(t_start);
    fprintf('%s: done in %.1f s (%d steps)\n', outtag, t_elapsed, numel(states));

    ns = numel(states);
    sg_all = zeros(G.cells.num, ns);
    rs_all = zeros(G.cells.num, ns);
    bhp_all = zeros(ns, 1);
    for n = 1:ns
        sg_all(:,n) = states{n}.s(:,2);
        rs_all(:,n) = states{n}.rs;
        bhp_all(n)  = wellSols{n}.bhp;
    end
    tvec = cumsum(schedule.step.val);
    centroids = G.cells.centroids;
    save('-v7', fullfile(resdir, ['diff_' outtag '.mat']), ...
         'sg_all', 'rs_all', 'bhp_all', 'tvec', 'centroids', 'fid', 'wellInx', ...
         't_elapsed', 'wellCell', 'wiModel', 'WI');
end

### A note on the well model

`addWell` (used above for the injector `W`) couples the well to the
reservoir with MRST's standard **Peaceman well model**. For each perforated
cell it computes an equivalent (Peaceman) radius from the cell's transverse
dimensions $d_x,d_y$ and permeabilities $k_x,k_y$,
$$
r_0 = \frac{0.28\sqrt{\sqrt{k_y/k_x}\,d_x^2 + \sqrt{k_x/k_y}\,d_y^2}}
           {(k_y/k_x)^{1/4}+(k_x/k_y)^{1/4}},
$$
and a well index
$$
WI = \frac{2\pi\sqrt{k_x k_y}\,h}{\ln(r_0/r_w) + \text{skin}}
$$
(`core/utils/computeWellIndex.m`; its internal validation routine is named
`check_peaceman_wi`, and `SimpleWell.m` documents its equations as
"Peaceman type"). During the simulation the well couples to the reservoir
purely algebraically, as a point source/sink: the perforation rate is
$q = -WI\cdot\mathrm{mob}\cdot(p_{\mathrm{cell}} - (p_{bh} + \Delta
p_{\mathrm{hydrostatic}}))$, with only a gravity (hydrostatic) correction
along the wellbore and no wellbore friction or flow dynamics — MRST has
separate multi-segment-well and near-wellbore modules for that, not used
here.

## Coarse mesh (8,206 cells)

Set `RUN_LIVE_COARSE = true` above to re-run both diffusivities from
scratch (about 5.6 min for the pair). By default this loads
`results/diff_D0.mat` and `results/diff_D2e11.mat`.

In [ ]:
if RUN_LIVE_COARSE
    run_diffusion_case('coarse', 0,    'D0',    RESDIR, FLUIDDECK);
    run_diffusion_case('coarse', 2e-11,'D2e11', RESDIR, FLUIDDECK);
end
d0 = load(fullfile(RESDIR, 'diff_D0.mat'));
d2 = load(fullfile(RESDIR, 'diff_D2e11.mat'));
fprintf('Loaded. Runtimes: D=0 %.1f s, D=2e-11 %.1f s\n', d0.t_elapsed, d2.t_elapsed);

At this resolution numerical diffusion ($D_{\mathrm{num}}\sim10^{-9}$
m2/s at $h=1$ cm) dominates the physical value, so the two diffusivities
give nearly identical fields.

In [ ]:
yq = linspace(min(d0.centroids(:,2)), max(d0.centroids(:,2)), 400);
zq = linspace(min(d0.centroids(:,3)), max(d0.centroids(:,3)), 260);
[Yq, Zq] = meshgrid(yq, zq);

cases4 = {d0, d2};
names = {'D = 0', 'D = 2e-11 m^2/s'};
figure('Position', [50 50 1200 700]);
for c = 1:2
    d = cases4{c};
    rs_end = d.rs_all(:, end);
    F = griddata(d.centroids(:,2), d.centroids(:,3), rs_end, Yq, Zq, 'nearest');
    subplot(2,1,c);
    imagesc(yq, zq, F);           % z axis: depth increases downward
    axis tight
    set(gca, 'YDir', 'reverse');
    colormap(flipud(bone)); colorbar;
    xlabel('y [m]'), ylabel('depth [m]')
    title(sprintf('%s   (max rs = %.4f, mean rs = %.5f)', names{c}, ...
          max(rs_end), mean(rs_end)));
end
print(gcf, fullfile(FIGDIR, 'fig_diffusion_rs30d.png'), '-dpng', '-r130');

fprintf('\n%-32s %-12s %-12s\n', 'Metric (t = 30 d)', 'D = 0', 'D = 2e-11');
fprintf('%-32s %-12.4f %-12.4f\n', 'Max rs', ...
        max(d0.rs_all(:,end)), max(d2.rs_all(:,end)));
fprintf('%-32s %-12d %-12d\n', 'Cells with rs > 1e-3', ...
        sum(d0.rs_all(:,end) > 1e-3), sum(d2.rs_all(:,end) > 1e-3));
fprintf('%-32s %-12.3f %-12.3f\n', 'Max Sg (free gas)', ...
        max(d0.sg_all(:,end)), max(d2.sg_all(:,end)));

## Fine mesh (105,279 cells)

Set `RUN_LIVE_FINE = true` above to re-run both diffusivities from scratch
(about 1.75 h for the pair — this is the most expensive case in the whole
reproduction). By default this loads `results/diff_fine_D0.mat` and
`results/diff_fine_D2e11.mat`.

At this resolution the convective fingers are resolved and the physical
diffusivity becomes visible: without diffusion the fingers below the gas
accumulation are thin and closely spaced; with $D=2\times10^{-11}$
m2/s they are wider, smoother, and farther apart. Compare with Fig. 6(a,b)
of the paper.

In [ ]:
if RUN_LIVE_FINE
    run_diffusion_case('fine', 0,    'fine_D0',    RESDIR, FLUIDDECK);
    run_diffusion_case('fine', 2e-11,'fine_D2e11', RESDIR, FLUIDDECK);
end
f0 = load(fullfile(RESDIR, 'diff_fine_D0.mat'));
f2 = load(fullfile(RESDIR, 'diff_fine_D2e11.mat'));
fprintf('Loaded. Runtimes: D=0 %.1f s, D=2e-11 %.1f s\n', f0.t_elapsed, f2.t_elapsed);

In [ ]:
yq = linspace(min(f0.centroids(:,2)), max(f0.centroids(:,2)), 700);
zq = linspace(min(f0.centroids(:,3)), max(f0.centroids(:,3)), 460);
[Yq, Zq] = meshgrid(yq, zq);

cases5 = {f0, f2};
figure('Position', [50 50 1200 700]);
for c = 1:2
    d = cases5{c};
    rs_end = d.rs_all(:, end);
    F = griddata(d.centroids(:,2), d.centroids(:,3), rs_end, Yq, Zq, 'nearest');
    subplot(2,1,c);
    imagesc(yq, zq, F);
    axis tight
    set(gca, 'YDir', 'reverse');
    colormap(flipud(bone)); colorbar;
    xlabel('y [m]'), ylabel('depth [m]')
    title(sprintf('%s   (max rs = %.4f, mean rs = %.5f)', names{c}, ...
          max(rs_end), mean(rs_end)));
end
print(gcf, fullfile(FIGDIR, 'fig_diffusion_fine30d.png'), '-dpng', '-r130');

fprintf('\n%-32s %-12s %-12s\n', 'Metric (t = 30 d, fine)', 'D = 0', 'D = 2e-11');
fprintf('%-32s %-12.4f %-12.4f\n', 'Max rs', ...
        max(f0.rs_all(:,end)), max(f2.rs_all(:,end)));
fprintf('%-32s %-12d %-12d\n', 'Cells with rs > 1e-3', ...
        sum(f0.rs_all(:,end) > 1e-3), sum(f2.rs_all(:,end) > 1e-3));
fprintf('%-32s %-12.5f %-12.5f\n', 'Mean rs', ...
        mean(f0.rs_all(:,end)), mean(f2.rs_all(:,end)));

## Summary

Diffusion enhances dissolution trapping: on the fine mesh the mean
dissolved ratio rises and the invaded region grows once the physical
diffusivity is resolved, while on the coarse mesh numerical diffusion
already masks the effect. This confirms the scaling argument above: the
physical diffusivity only matters once the grid-level numerical diffusion
drops below it.

## Appendix: is the injection well model really Peaceman?

The note on the well model above identified MRST's coupling as a
**Peaceman-type** model. This appendix asks a sharper question: Peaceman's
own formula was derived for a rectangular Cartesian block — is it still
correct on a PEBI/Voronoi cell such as this diffusion case's well cell,
and if not, by how much, and does it matter for the simulated result?

### A.1 The well-reservoir coupling used by MRST

For a single perforated cell, MRST's rate/pressure relation and well
index are (`core/utils/computeWellIndex.m`)
$$
q = -WI\cdot\mathrm{mob}\cdot\big(p_{\mathrm{cell}} - p_{bh} - \Delta
p_{\mathrm{hydrostatic}}\big), \qquad
WI = \frac{2\pi\sqrt{k_x k_y}\,h}{\ln(r_0/r_w) + \mathrm{skin}},
$$
with the anisotropic Peaceman equivalent radius (Peaceman, 1983)
$$
r_0 = \frac{0.28\sqrt{\sqrt{k_y/k_x}\,\Delta x^2 +
\sqrt{k_x/k_y}\,\Delta y^2}}{(k_y/k_x)^{1/4}+(k_x/k_y)^{1/4}}.
$$
This formula assumes the well cell **is** a rectangle of known
$\Delta x,\Delta y$ — true by construction on a Cartesian or corner-point
grid, but not defined at all for an irregular PEBI polygon.

### A.2 Why this does not transfer cleanly to a PEBI grid

For any grid with a `nodes` field, MRST's `cellDims.m` supplies $\Delta
x,\Delta y$ for `computeWellIndex` by taking the **axis-aligned bounding
box** of the cell's own nodes, whatever its true shape:
```matlab
nodes  = unique(G.faces.nodes(e, 1));
coords = G.nodes.coords(nodes,:);
m = min(coords);  M = max(coords);
dx(k) = M(1) - m(1);  dy(k) = M(2) - m(2);
```
This never looks at the well cell's actual neighbors, their individual
transmissibilities, or how many of them there are — an irregular hexagon
and a very different-shaped cell with the same bounding box get the same
well index.

### A.3 Palagi & Aziz's well index for Voronoi grids

Palagi and Aziz (SPE 22889, 1991; SPE 24072, 1994) derive an equivalent
radius directly from the well cell's **actual connections**, so it is
defined for any polygon. Assuming radial flow around the well-cell
gridpoint out to each neighbor $j$ (SPE 24072, Eq. 15),
$$
p_j - p_w = \frac{qB\mu\,\ln(L_{ij}/r_w)}{\theta kh},
$$
and requiring this to be consistent with the discretized single-phase
flow equation for the well cell (Eq. 16),
$$
q = \sum_j \frac{T_{ij}}{B\mu}(p_j - p_o)
  = \sum_j \frac{T_{ij}}{B\mu}(p_j - p_w) - \sum_j
    \frac{T_{ij}}{B\mu}(p_o - p_w),
$$
gives the equivalent radius as a **transmissibility-weighted average over
the real connections** (Eq. 17):
$$
\ln r_{eq} = \frac{\sum_j T_{ij}\ln L_{ij} - \theta kh}{\sum_j T_{ij}},
\qquad
WI = \frac{\theta kh}{\ln(r_{eq}/r_w) + \mathrm{skin}},
$$
where $T_{ij}$ is the two-point transmissibility of connection $ij$,
$L_{ij}$ the distance between cell centroids, and $\theta$ the angle open
to flow ($\theta=2\pi$ for a fully interior well cell — the same value
MRST's own `2*pi` in the formula above already assumes, which keeps the
two models directly comparable). This reduces to Peaceman's formula for a
square Cartesian cell, but stays correct for an irregular polygon with any
number of neighbors.

In [ ]:
type('/home/adriano/codes/MRST/reproductions/salo2024-gcs/scripts/computeWellIndexPalagiAziz.m');

### A.4 Comparing the two well indices

Both models are evaluated at radius $r_w=$ 1 mm and skin $=0$, with
identical $Kh$, so they differ **only** in how $r_{eq}$ is derived. Two
locations are compared:

- The **actual injection cell** used above (in the reservoir matrix,
  nearly a cube: $\Delta x\approx\Delta y\approx\Delta z\approx1$ cm).
- The **most elongated fault-strip cell reachable near that same location**
  (bounding-box aspect ratio $R_y\approx2.4$; the ranking that found it is
  in `scripts/find_fault_cell.m`). Palagi & Aziz's own paper (SPE 24072)
  flags $R_y>2$ as exactly the regime where the Cartesian formula stops
  being reliable.

In [ ]:
G_dat = simpleExtrudedFluidFlowerMesh('coarse');   % cache hit, a few seconds
sealID = [1 3 5 8];
faultID = 6;
G = G_dat.G;
if isfield(G.faces, 'tag'), G.faces = rmfield(G.faces, 'tag'); end
[G, cellmap] = removeCells(G, ismember(G_dat.compartID, sealID));
cidAll = 1:G_dat.G.cells.num;
fidAll = cidAll(G_dat.compartID == faultID);
fidApp = ismember(cellmap, fidAll);
rockApp.poro = 0.3*ones(G.cells.num, 1); rockApp.poro(fidApp) = 0.1;
rockApp.perm = 1000*ones(G.cells.num, 1); rockApp.perm(fidApp) = 10;
rockApp.perm = rockApp.perm*(milli*darcy);

radiusApp = 1e-3;
dist = sqrt(sum((G.cells.centroids - [0.005, 0.903, 1000.58]).^2, 2));
[~, wellRes] = min(dist);
wellFault = 555;   % elongated fault-strip cell (Ry ~ 2.4), see A.4 above

cellsApp = [wellRes, wellFault];
namesApp = {'Reservoir well (original)', 'Elongated fault-strip cell'};
fprintf('%-28s %-6s %-14s %-14s %-10s\n', ...
        'Location', 'nConn', 'WI Peaceman', 'WI Palagi-Aziz', 'diff [%]');
for i = 1:2
    cc = cellsApp(i);
    WIp = computeWellIndex(G, rockApp, radiusApp, cc, 'Dir', 'z');
    [WIpa, infoApp] = computeWellIndexPalagiAziz(G, rockApp, cc, radiusApp, 'Dir', 'z');
    fprintf('%-28s %-6d %-14.4e %-14.4e %-10.2f\n', ...
            namesApp{i}, infoApp.nConn, WIp, WIpa, 100*(WIpa-WIp)/WIp);
end

The elongated fault cell shows a substantially larger gap than the
well-conditioned reservoir cell — consistent with Peaceman's bounding-box
shortcut degrading specifically where the cell departs from a square, not
uniformly everywhere.

### A.5 Does the well-index choice change the simulated result?

Because this well is **rate-controlled** (`'Type', 'rate'`), $WI$ only sets
the drawdown, $p_{cell}-p_{bh}$, needed to deliver the fixed 8 mL/min — it
never changes how much CO2 enters the reservoir. So the dissolved/free-gas
fields should be governed by rate, not by $WI$, and only the bottom-hole
pressure should respond. The cell below tests this directly: for both well
locations, it reruns the coarse-mesh, $D=0$ case with each of the two well
indices (four runs total, `run_diffusion_case`'s optional `wellCell`/
`wiModel` arguments), and loads the cached results by default.

Runtime if `RUN_LIVE_WI = true`: about 4 x 180 s = 12 min.

In [ ]:
RUN_LIVE_WI = false;

In [ ]:
wiRunSpecs = { ...
    {wellRes,   'peaceman',    'D0_wi_peaceman'}, ...
    {wellRes,   'palagi-aziz', 'D0_wi_palagiaziz'}, ...
    {wellFault, 'peaceman',    'D0_wi_peaceman_fault'}, ...
    {wellFault, 'palagi-aziz', 'D0_wi_palagiaziz_fault'} ...
};
if RUN_LIVE_WI
    for i = 1:numel(wiRunSpecs)
        spec = wiRunSpecs{i};
        run_diffusion_case('coarse', 0, spec{3}, RESDIR, FLUIDDECK, spec{1}, spec{2});
    end
end

wp_res  = load(fullfile(RESDIR, 'diff_D0_wi_peaceman.mat'));
wpa_res = load(fullfile(RESDIR, 'diff_D0_wi_palagiaziz.mat'));
wp_flt  = load(fullfile(RESDIR, 'diff_D0_wi_peaceman_fault.mat'));
wpa_flt = load(fullfile(RESDIR, 'diff_D0_wi_palagiaziz_fault.mat'));

fprintf('%-28s %-16s %-16s %-16s\n', 'Location', 'Max |dBHP| [Pa]', ...
        'Max |d(rs)|', 'Max |d(sg)|');
fprintf('%-28s %-16.4e %-16.4e %-16.4e\n', 'Reservoir well', ...
        max(abs(wp_res.bhp_all - wpa_res.bhp_all)), ...
        max(abs(wp_res.rs_all(:) - wpa_res.rs_all(:))), ...
        max(abs(wp_res.sg_all(:) - wpa_res.sg_all(:))));
fprintf('%-28s %-16.4e %-16.4e %-16.4e\n', 'Fault-strip cell', ...
        max(abs(wp_flt.bhp_all - wpa_flt.bhp_all)), ...
        max(abs(wp_flt.rs_all(:) - wpa_flt.rs_all(:))), ...
        max(abs(wp_flt.sg_all(:) - wpa_flt.sg_all(:))));

### A.6 Conclusion

MRST's well model is Peaceman-type, but the equivalent radius it uses for
a PEBI cell is a crude bounding-box stand-in for the real Cartesian
formula, not a grid-aware generalization like Palagi & Aziz's. That gap is
real and grows with cell irregularity (about twice as large at an
elongated fault-strip cell as at a well-conditioned reservoir cell in this
mesh). Whether it matters for a given simulation is a separate question:
for this rate-controlled injector, it changes only the computed bottom-hole
pressure, never the injected mass, and at this problem's flow rate the
absolute pressure change is small even where the well-index gap is
largest — the reservoir has far more injectivity headroom than this well
ever calls on. A pressure-controlled well, a higher rate, or a
lower-permeability target would be expected to show a larger effect,
since drawdown itself would no longer be negligible.